# Шаг 2. Языковые модели и гибридный детектор

Что делает ноутбук (≈30–50 минут):
1. Берёт таблицу текстов из шага 1 (`dataset_step1.parquet` на Диске) и русский корпус CoAT (RuATD).
2. Прогоняет тексты через компактные языковые модели — **GPT-2 small** (английский) и **ruGPT-3 small** (русский) — и считает вероятностные признаки.
3. Обучает три детектора: стилометрический, статистический (только признаки модели) и **гибридный**.
4. Эксперименты: Э1 — качество по языкам, Э3 — перенос M4 → RuATD, Э4 — зависимость от длины текста, Э5 — атаки гомоглифами и опечатками.
5. Сохраняет на Диск `results_step2.json`, `weights_hybrid.json` и графики.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU** → Сохранить. Затем «Выполнить все».
**Прислать:** вывод ячеек 4–8 и файлы `results_step2.json`, `weights_hybrid.json`, `length_curve.png`.

## 1. Подготовка

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, torch
WORK = '/content/drive/MyDrive/ai-text-detector'
os.makedirs('/content/code', exist_ok=True); os.chdir('/content/code'); sys.path.insert(0, '/content/code')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '— ВНИМАНИЕ: видеокарта не подключена, будет очень медленно')

## 2. Код проекта

In [ ]:
%%writefile /content/code/stylometry.py
"""Стилометрические признаки (та же логика будет реализована в расширении на JavaScript)."""
import re, math, unicodedata
from collections import Counter

RU_FUNC = set("и в во не на я что он с со как а то все она так его но да ты к ко у же вы за бы по только ее её мне было вот от меня еще ещё нет о об из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее неё сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между".split())
EN_FUNC = set("the of and to a in is that it for on as with was be by this are or from at an but not have has had which they you he she we his her their its can will would there been were if all so no one more also do than about into such these other only some may then".split())

CYR2LAT = {"А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H","О":"O","Р":"P","С":"C","Т":"T","Х":"X",
           "а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x"}
LAT2CYR = {v: k for k, v in CYR2LAT.items()}

def cyr_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('Ѐ' <= ch <= 'ӿ' for ch in letters) / len(letters)

def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return 0.0
    return sum('a' <= ch.lower() <= 'z' for ch in letters) / len(letters)

def detect_lang(text):
    """'ru' — преобладает кириллица, 'en' — почти только латиница, иначе 'other' (не поддерживается)."""
    if cyr_ratio(text) > 0.5: return 'ru'
    if latin_ratio(text) > 0.9: return 'en'
    return 'other'

def _fix_word(word):
    """Слово из смеси алфавитов приводим к алфавиту большинства букв (защита от атаки гомоглифами)."""
    cyr = sum('\u0400' <= ch <= '\u04FF' for ch in word)
    lat = sum('a' <= ch.lower() <= 'z' for ch in word)
    if cyr and lat:
        table = LAT2CYR if cyr >= lat else CYR2LAT
        return ''.join(table.get(ch, ch) for ch in word)
    return word

def normalize(text):
    """NFKC, удаление невидимых символов, нормализация пробелов, исправление гомоглифов."""
    t = unicodedata.normalize('NFKC', text)
    t = re.sub(r'[\u200b\u200c\u200d\u2060\ufeff]', '', t)
    t = re.sub(r'[ \t\u00a0]+', ' ', t)
    t = re.sub(r'\s*\n\s*', '\n', t).strip()
    return re.sub(r'\w+', lambda m: _fix_word(m.group(0)), t)

SENT_SPLIT = re.compile(r'(?<=[.!?…])\s+')
WORD = re.compile(r"[^\W\d_]+(?:[-'’][^\W\d_]+)*", re.UNICODE)

FEATURE_NAMES = ['mean_sent_len', 'cv_sent_len', 'mean_word_len', 'ttr', 'punct_ratio',
                 'func_ratio', 'rep_bigram_ratio', 'rep_trigram_ratio', 'comma_per_sent']

def stylometric_features(text, lang=None):
    lang = lang or detect_lang(text)
    sents = [s for s in SENT_SPLIT.split(text) if s.strip()]
    words = [w.lower() for w in WORD.findall(text)]
    n = max(len(words), 1)
    slens = [len(WORD.findall(s)) for s in sents] or [0]
    mu = sum(slens) / len(slens)
    sd = math.sqrt(sum((x - mu) ** 2 for x in slens) / len(slens))
    func = RU_FUNC if lang == 'ru' else EN_FUNC
    def rep(k):
        grams = [tuple(words[i:i+k]) for i in range(len(words) - k + 1)]
        if not grams: return 0.0
        c = Counter(grams); return sum(v for v in c.values() if v > 1) / len(grams)
    chars = max(len(text), 1)
    return {
        'mean_sent_len': mu,
        'cv_sent_len': sd / mu if mu else 0.0,
        'mean_word_len': sum(map(len, words)) / n,
        'ttr': len(set(words)) / n,
        'punct_ratio': sum(unicodedata.category(ch).startswith('P') for ch in text) / chars,
        'func_ratio': sum(w in func for w in words) / n,
        'rep_bigram_ratio': rep(2),
        'rep_trigram_ratio': rep(3),
        'comma_per_sent': text.count(',') / max(len(sents), 1),
    }


In [ ]:
%%writefile /content/code/data_loading.py
"""Поиск и загрузка данных M4 и RuATD в единую таблицу:
text | label (1 = ИИ, 0 = человек) | lang | source | generator | pair_id"""
import os, json, glob, hashlib
import pandas as pd
from stylometry import detect_lang

HUMAN_KEYS = ['human_text', 'human', 'text_human', 'original_text']
MACHINE_KEYS = ['machine_text', 'machine', 'text_machine', 'generated_text', 'ai_text']
TEXT_KEYS = ['text', 'Text', 'content']
LABEL_KEYS = ['label', 'Label', 'Class', 'class', 'labels']
MODEL_KEYS = ['model', 'Model', 'generator', 'source_model']

def _first(d, keys):
    for k in keys:
        if k in d and d[k] not in (None, ''):
            return k
    return None

def _read_records(path, limit=None):
    recs = []
    with open(path, encoding='utf-8') as f:
        head = f.read(1); f.seek(0)
        if head == '[':                       # обычный JSON-массив
            data = json.load(f); recs = data[:limit] if limit else data
        else:                                 # JSON Lines
            for line in f:
                line = line.strip()
                if not line: continue
                try: recs.append(json.loads(line))
                except json.JSONDecodeError: continue
                if limit and len(recs) >= limit: break
    return recs

def _pid(*parts):
    return hashlib.md5('|'.join(map(str, parts)).encode()).hexdigest()[:12]

def _label_value(v):
    s = str(v).strip().lower()
    if s in ('1', 'm', 'machine', 'ai', 'generated', 'fake', 'true'): return 1
    if s in ('0', 'h', 'human', 'real', 'false'): return 0
    return None

def inspect_files(root, exts=('.jsonl', '.json', '.csv', '.tsv'), max_files=200):
    """Печатает список файлов данных и ключи первой записи — для диагностики."""
    files = [p for p in glob.glob(os.path.join(root, '**', '*'), recursive=True)
             if p.lower().endswith(exts) and os.path.getsize(p) > 0]
    print(f'{root}: найдено файлов данных — {len(files)}')
    for p in sorted(files)[:max_files]:
        size = os.path.getsize(p) / 1e6
        try:
            if p.lower().endswith(('.csv', '.tsv')):
                cols = list(pd.read_csv(p, sep='\t' if p.endswith('.tsv') else ',', nrows=2).columns)
            else:
                r = _read_records(p, limit=1); cols = list(r[0].keys()) if r else []
        except Exception as e:
            cols = [f'ошибка чтения: {e}']
        print(f'  {os.path.relpath(p, root)}  ({size:.1f} МБ)  поля: {cols[:12]}')
    return files

def load_json_dir(root, dataset_name, limit_per_file=None):
    rows = []
    for p in glob.glob(os.path.join(root, '**', '*.json*'), recursive=True):
        try: recs = _read_records(p, limit=limit_per_file)
        except Exception: continue
        fname = os.path.splitext(os.path.basename(p))[0]
        for i, r in enumerate(recs):
            if not isinstance(r, dict): continue
            gen = r.get(_first(r, MODEL_KEYS)) if _first(r, MODEL_KEYS) else fname
            hk, mk = _first(r, HUMAN_KEYS), _first(r, MACHINE_KEYS)
            if hk or mk:                                   # параллельный формат (как в M4)
                pid = _pid(dataset_name, r[hk]) if hk else _pid(dataset_name, fname, i)  # пара = один человеческий текст
                if hk: rows.append((r[hk], 0, fname, 'human', pid))
                if mk:
                    m = r[mk]
                    for j, t in enumerate(m if isinstance(m, list) else [m]):
                        rows.append((t, 1, fname, str(gen), pid))
                continue
            tk, lk = _first(r, TEXT_KEYS), _first(r, LABEL_KEYS)
            if tk and lk is not None and _label_value(r[lk]) is not None:
                rows.append((r[tk], _label_value(r[lk]), fname, str(gen) if _label_value(r[lk]) else 'human', _pid(dataset_name, fname, i)))
    return _finish(rows, dataset_name)

def load_csv_dir(root, dataset_name):
    rows = []
    for p in glob.glob(os.path.join(root, '**', '*.[ct]sv'), recursive=True):
        try: df = pd.read_csv(p, sep='\t' if p.endswith('.tsv') else ',')
        except Exception: continue
        tk = _first({c: 1 for c in df.columns}, TEXT_KEYS)
        lk = _first({c: 1 for c in df.columns}, LABEL_KEYS)
        if not tk or not lk: continue
        fname = os.path.splitext(os.path.basename(p))[0]
        for i, (t, l) in enumerate(zip(df[tk], df[lk])):
            lab = _label_value(l)
            if lab is None:                  # многоклассовая разметка: 'Human' = 0, имя модели = 1
                lab = 0 if str(l).strip().lower() == 'human' else 1
                gen = 'human' if lab == 0 else str(l)
            else:
                gen = 'human' if lab == 0 else 'unknown'
            rows.append((t, lab, fname, gen, _pid(dataset_name, fname, i)))
    return _finish(rows, dataset_name)

# Язык M4 определяем по имени файла: кириллицей пишутся и болгарский, и казахский,
# латиницей — немецкий и индонезийский, поэтому автоопределение по алфавиту здесь ненадёжно.
M4_LANG_BY_PREFIX = {'russian': 'ru', 'arxiv': 'en', 'peerread': 'en', 'reddit': 'en', 'wikihow': 'en', 'wikipedia': 'en'}
BLOOMZ_HUMAN_KEYS = {'arxiv': 'abstract', 'wikihow': 'text', 'wikipedia': 'text', 'reddit': 'text'}

def load_m4(root):
    """Загрузка английской и русской частей M4 с учётом разных форматов файлов."""
    rows = []
    for p in sorted(glob.glob(os.path.join(root, 'data', '*.jsonl'))):
        fname = os.path.splitext(os.path.basename(p))[0]
        prefix, gen_from_name = fname.split('_')[0], fname.split('_')[-1]
        lang = M4_LANG_BY_PREFIX.get(prefix)
        if lang is None:
            continue
        for i, r in enumerate(_read_records(p)):
            gen = str(r.get('model') or gen_from_name)
            human = r.get('human_text') or (r.get(BLOOMZ_HUMAN_KEYS.get(prefix, '')) if 'bloomz' in fname else None)
            machine = r.get('machine_text')
            if prefix == 'peerread' and 'bloomz' in fname:          # отзывы хранятся списками
                human = (r.get('human_reviews') or [None])[0]
                machine = (r.get('bloom_reviews') or [None])[0]
                gen = 'bloomz'
            pid = _pid('M4', human) if isinstance(human, str) else _pid('M4', fname, i)
            if isinstance(human, str): rows.append((human, 0, fname, 'human', pid, lang))
            for t in (machine if isinstance(machine, list) else [machine]):
                if isinstance(t, str): rows.append((t, 1, fname, gen, pid, lang))
    df = pd.DataFrame(rows, columns=['text', 'label', 'source', 'generator', 'pair_id', 'lang'])
    df = df[df.text.str.strip().str.len() > 0]
    df['label'] = df.label.astype(int); df['dataset'] = 'M4'
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)

def load_coat(config='binary'):
    """Корпус CoAT (опубликованная версия RuATD) с Hugging Face. Столбцы определяются автоматически."""
    from datasets import load_dataset
    ds = load_dataset('RussianNLP/coat', config)
    print(ds)
    frames = []
    for split in ds:
        d = ds[split].to_pandas()
        feats = ds[split].features
        text_col = max([c for c in d.columns if d[c].dtype == object or pd.api.types.is_string_dtype(d[c])],
                       key=lambda c: d[c].astype(str).str.len().mean())
        label_col = None
        for c in d.columns:
            if c == text_col: continue
            names = getattr(feats[c], 'names', None)
            vals = names or sorted(map(str, d[c].dropna().unique()))[:50]
            if any(str(v).lower() in ('h', 'human', 'm', 'machine') for v in vals):
                label_col = c; break
        if label_col is None:
            print(f'[{split}] столбец с меткой не найден, столбцы: {list(d.columns)}'); continue
        names = getattr(feats[label_col], 'names', None)
        def to_name(v):
            if names and str(v).isdigit() and int(v) < len(names): return names[int(v)]
            return v
        lab = d[label_col].map(to_name)
        lab = lab.astype(str).str.lower()
        if lab.isin(['h', 'm', 'human', 'machine']).all():
            y = lab.isin(['m', 'machine']).astype(int); gen = ['human' if v == 0 else 'unknown' for v in y]
        else:                                     # многоклассовая разметка: human или имя модели
            y = (lab != 'human').astype(int); gen = lab.tolist()
        frames.append(pd.DataFrame({'text': d[text_col], 'label': y, 'source': f'coat_{split}', 'generator': gen,
                                    'pair_id': [_pid('coat', split, i) for i in range(len(d))], 'lang': 'ru'}))
        print(f'[{split}] текст: {text_col}, метка: {label_col}, примеры меток: {lab.unique()[:8]}')
    if not frames: return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True); df['dataset'] = 'RuATD'
    df = df[df.text.astype(str).str.strip().str.len() > 0]
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)

def _finish(rows, dataset_name):
    df = pd.DataFrame(rows, columns=['text', 'label', 'source', 'generator', 'pair_id'])
    if df.empty: return df
    df = df[df.text.apply(lambda t: isinstance(t, str) and len(t.strip()) > 0)].copy()
    df['lang'] = df.text.apply(detect_lang)
    df['label'] = df.label.astype(int)
    df['dataset'] = dataset_name
    return df.drop_duplicates(['text', 'label']).reset_index(drop=True)


In [ ]:
%%writefile /content/code/metrics_utils.py
"""Метрики качества детектора (раздел 1.6)."""
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, precision_score, recall_score

def tpr_at_fpr(y, p, fpr_target=0.01):
    fpr, tpr, _ = roc_curve(y, p)
    ok = fpr <= fpr_target
    return float(tpr[ok].max()) if ok.any() else 0.0

def evaluate(y, p, threshold=0.5, n_boot=200, seed=0):
    y, p = np.asarray(y), np.asarray(p); yhat = (p >= threshold).astype(int)
    res = {'n': int(len(y)), 'F1': f1_score(y, yhat), 'Precision': precision_score(y, yhat, zero_division=0),
           'Recall': recall_score(y, yhat), 'FPR': float(((yhat == 1) & (y == 0)).sum() / max((y == 0).sum(), 1)),
           'AUROC': roc_auc_score(y, p), 'TPR@1%FPR': tpr_at_fpr(y, p)}
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):                       # доверительный интервал AUROC (бутстреп)
        idx = rng.integers(0, len(y), len(y))
        if len(set(y[idx])) == 2: boots.append(roc_auc_score(y[idx], p[idx]))
    res['AUROC_95CI'] = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))) if boots else None
    return res


In [ ]:
%%writefile /content/code/baseline_stylometry.py
"""Базовый стилометрический детектор: признаки + логистическая регрессия, отдельно для каждого языка."""
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from stylometry import stylometric_features, normalize, FEATURE_NAMES
from metrics_utils import evaluate

def _features_with_format(text, lang):
    t = normalize(text)
    f = stylometric_features(t, lang)
    f['newline_ratio'] = t.count('\n') / max(len(t), 1)   # признак форматирования — только для эксперимента-абляции
    return f

def featurize(df, names=FEATURE_NAMES):
    return np.array([[f[k] for k in names] for f in
                     (_features_with_format(t, l) for t, l in zip(df.text, df.lang))])

def split(df, seed=42):
    """70/15/15 с разбиением по pair_id: человеческий текст и его сгенерированная пара не разделяются."""
    g1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    tr, rest = next(g1.split(df, groups=df.pair_id))
    rest_df = df.iloc[rest]
    g2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    va, te = next(g2.split(rest_df, groups=rest_df.pair_id))
    return df.iloc[tr], rest_df.iloc[va], rest_df.iloc[te]

def run(df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    d = df[df.lang == lang]
    d = pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])
    if d.label.nunique() < 2:
        return None, None, None
    tr, va, te = split(d, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'val': len(va), 'test': len(te)})
    coefs = pd.Series(model[-1].coef_[0], index=names).sort_values(key=abs, ascending=False)
    return res, coefs, model

def export_weights(models, path):
    """Сохраняет параметры нормировки и логистической регрессии в JSON для расширения (extension/model/weights.json)."""
    import json
    out = {'version': 'stylometry-v1', 'demo': False, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        out['langs'][lang] = {'features': FEATURE_NAMES, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0])}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=1)

def balanced_sample(d, cap_per_class, seed=42):
    return pd.concat([g.sample(min(len(g), cap_per_class), random_state=seed) for _, g in d.groupby('label')])

def run_cross(train_df, test_df, lang, cap_per_class=20000, seed=42, names=FEATURE_NAMES):
    """Обучение на одном корпусе и проверка на другом (перенос на незнакомые генераторы, серия Э3)."""
    tr = balanced_sample(train_df[train_df.lang == lang], cap_per_class, seed)
    te = balanced_sample(test_df[test_df.lang == lang], cap_per_class // 4, seed)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced'))
    model.fit(featurize(tr, names), tr.label)
    res = evaluate(te.label.values, model.predict_proba(featurize(te, names))[:, 1])
    res.update({'lang': lang, 'train': len(tr), 'test': len(te)})
    return res


In [ ]:
%%writefile /content/code/lm_features.py
"""Вероятностные признаки по компактной авторегрессионной языковой модели (раздел 2.3, формулы 2.1–2.2).
Для каждого токена считаются: логарифм вероятности, энтропия предсказанного распределения, ранг
фактического токена и дисперсия логарифма вероятности (для признака условной кривизны, как в Fast-DetectGPT)."""
import numpy as np

LM_FEATURE_NAMES = ['mean_lp', 'std_lp', 'mean_entropy', 'mean_log_rank',
                    'rank_1', 'rank_2_10', 'rank_11_100', 'rank_101_1000', 'rank_gt_1000', 'curvature']

def token_stats(texts, model, tok, max_tokens=256, batch_size=8, device='cuda'):
    """Один проход модели по каждому тексту. Возвращает список словарей с массивами по токенам."""
    import torch
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    out = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors='pt', truncation=True, max_length=max_tokens,
                  padding=True, add_special_tokens=False).to(device)
        with torch.no_grad():
            logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask).logits.float()
        logp = torch.log_softmax(logits[:, :-1], dim=-1)          # прогноз для токенов 2..T
        target = enc.input_ids[:, 1:]
        lp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)    # log P(w_i | w_<i)
        p = logp.exp()
        ent = -(p * logp).sum(-1)                                  # энтропия, формула (2.2)
        var = (p * logp.pow(2)).sum(-1) - ent.pow(2)               # дисперсия log p по распределению модели
        rank = (logp > lp.unsqueeze(-1)).sum(-1) + 1               # ранг фактического токена
        mask = enc.attention_mask[:, 1:].bool()
        for b in range(len(batch)):
            m = mask[b]
            out.append({'ids': enc.input_ids[b][enc.attention_mask[b].bool()].cpu().numpy(),
                        'lp': lp[b][m].cpu().numpy(), 'ent': ent[b][m].cpu().numpy(),
                        'var': var[b][m].cpu().numpy(), 'rank': rank[b][m].cpu().numpy()})
        del logits, logp, p
    return out

def aggregate(s, k=None):
    """Признаки по первым k предсказанным токенам (k=None — все)."""
    lp, ent, var, rank = (s[x][:k] if k else s[x] for x in ('lp', 'ent', 'var', 'rank'))
    n = len(lp)
    if n == 0:
        return {f: 0.0 for f in LM_FEATURE_NAMES}
    curv = (lp.sum() + ent.sum()) / np.sqrt(max(var.sum(), 1e-8))   # (∑log p − ∑E[log p]) / √∑Var, E[log p] = −H
    return {
        'mean_lp': float(lp.mean()), 'std_lp': float(lp.std()), 'mean_entropy': float(ent.mean()),
        'mean_log_rank': float(np.log(rank).mean()),
        'rank_1': float((rank == 1).mean()), 'rank_2_10': float(((rank >= 2) & (rank <= 10)).mean()),
        'rank_11_100': float(((rank >= 11) & (rank <= 100)).mean()),
        'rank_101_1000': float(((rank >= 101) & (rank <= 1000)).mean()), 'rank_gt_1000': float((rank > 1000).mean()),
        'curvature': float(curv),
    }


In [ ]:
%%writefile /content/code/attacks.py
"""Простые атаки на детекторы (серия Э5): замена букв гомоглифами и внесение опечаток."""
import random, re
from stylometry import CYR2LAT, LAT2CYR

def homoglyph_attack(text, rate=0.15, seed=0):
    """Заменяет долю rate подходящих букв визуально похожими буквами другого алфавита."""
    rnd = random.Random(seed)
    table = {**CYR2LAT, **LAT2CYR}
    return ''.join(table[ch] if ch in table and rnd.random() < rate else ch for ch in text)

def typo_attack(text, rate=0.1, seed=0):
    """В доле rate слов длиной от 4 букв переставляет две соседние буквы."""
    rnd = random.Random(seed)
    def swap(m):
        w = m.group(0)
        if len(w) < 4 or rnd.random() >= rate: return w
        i = rnd.randrange(1, len(w) - 2)
        return w[:i] + w[i + 1] + w[i] + w[i + 2:]
    return re.sub(r'\w+', swap, text)


In [ ]:
%%writefile /content/code/hybrid.py
"""Обучение и оценка статистического, стилометрического и гибридного детекторов (разделы 2.3, 3.3)."""
import json
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from stylometry import stylometric_features, normalize, FEATURE_NAMES as STYLO
from lm_features import aggregate, LM_FEATURE_NAMES as LM
from metrics_utils import evaluate

VARIANTS = {'stylometric': STYLO, 'statistical': LM, 'hybrid': STYLO + LM}

def feature_table(texts, langs, stats, k=None, decode=None):
    """Признаки обеих групп. Если задан k, стилометрия считается по тексту первых k+1 токенов."""
    rows = []
    for t, l, s in zip(texts, langs, stats):
        if k is not None and decode is not None:
            t = decode(s['ids'][:k + 1], l)
        f = stylometric_features(t, l)
        f.update(aggregate(s, k))
        rows.append(f)
    return pd.DataFrame(rows)

def fit(X, y):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced'))
    return m.fit(X, y)

def threshold_at_fpr(model, X_val, y_val, fpr):
    """Порог, при котором на валидационной выборке доля ложных срабатываний равна fpr."""
    scores = model.predict_proba(X_val)[:, 1][np.asarray(y_val) == 0]
    return float(np.quantile(scores, 1 - fpr))

def train_eval_all(F_tr, y_tr, F_te, y_te, variants=VARIANTS):
    out, models = {}, {}
    for name, cols in variants.items():
        m = fit(F_tr[cols].values, y_tr)
        out[name] = evaluate(y_te, m.predict_proba(F_te[cols].values)[:, 1])
        models[name] = m
    return out, models

def export(models, thresholds, path, meta):
    """Веса гибридного детектора для расширения."""
    res = {'version': 'hybrid-v1', 'demo': False, 'meta': meta, 'langs': {}}
    for lang, m in models.items():
        sc, lr = m[0], m[-1]
        res['langs'][lang] = {'features': STYLO + LM, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
                              'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0]),
                              'thresholds': thresholds[lang]}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=1)


## 3. Данные

In [ ]:
import importlib, numpy as np, pandas as pd
import stylometry, data_loading, baseline_stylometry, lm_features, attacks, hybrid
for m in (stylometry, data_loading, baseline_stylometry, lm_features, attacks, hybrid): importlib.reload(m)
from stylometry import normalize
from data_loading import load_coat
from baseline_stylometry import split

SEED = 42
df = pd.read_parquet(f'{WORK}/dataset_step1.parquet')
df = df[df.dataset == 'M4']
try:
    coat = load_coat('binary')
    coat['n_words'] = coat.text.astype(str).str.split().str.len()
    print('CoAT: всего', len(coat), '| из них от 30 слов:', int((coat.n_words >= 30).sum()))
    coat = coat[coat.n_words >= 30]
except Exception as e:
    print('CoAT не загрузился:', repr(e)); coat = pd.DataFrame(columns=df.columns)

def bal(d, cap):
    return pd.concat([g.sample(min(len(g), cap), random_state=SEED) for _, g in d.groupby('label')])
data = {
    'en': bal(df[df.lang == 'en'], 6000),
    'ru': df[df.lang == 'ru'],
    'coat': bal(coat, 3000) if len(coat) else coat,
}
for k, d in data.items():
    d = d.copy(); d['norm'] = d.text.astype(str).map(normalize); data[k] = d.reset_index(drop=True)
    print(k, len(d), d.label.value_counts().to_dict())

## 4. Прогон через языковые модели

In [ ]:
import pickle, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from lm_features import token_stats
LM_NAMES = {'en': 'gpt2', 'ru': 'ai-forever/rugpt3small_based_on_gpt2'}
TOK, MODELS = {}, {}
def get_lm(lang):
    if lang not in MODELS:
        TOK[lang] = AutoTokenizer.from_pretrained(LM_NAMES[lang])
        MODELS[lang] = AutoModelForCausalLM.from_pretrained(LM_NAMES[lang]).to(DEVICE)
        n = sum(p.numel() for p in MODELS[lang].parameters())
        print(f'{LM_NAMES[lang]}: {n/1e6:.0f} млн параметров')
    return TOK[lang], MODELS[lang]
def get_stats(texts, lang):
    tok, model = get_lm(lang)
    return token_stats(list(texts), model, tok, max_tokens=256, batch_size=8, device=DEVICE)
def decode(ids, lang):
    tok, _ = get_lm(lang)
    return tok.decode(ids)

STATS = {}
cache = f'{WORK}/lm_stats_step2.pkl'
if os.path.exists(cache):
    STATS = pickle.load(open(cache, 'rb')); print('Загружено из кэша:', {k: len(v) for k, v in STATS.items()})
for key, d in data.items():
    if key in STATS or not len(d): continue
    lang = 'ru' if key in ('ru', 'coat') else 'en'
    t0 = time.time(); STATS[key] = get_stats(d.norm, lang)
    print(f'{key}: {len(d)} текстов за {time.time()-t0:.0f} с')
    pickle.dump(STATS, open(cache, 'wb'))

## 5. Э1 — качество трёх детекторов и экспорт гибридного

In [ ]:
from hybrid import feature_table, train_eval_all, threshold_at_fpr, export, VARIANTS
results = {'E1': {}, 'E3': {}, 'E4': {}, 'E5': {}}
SPLITS, FEATS, HYB = {}, {}, {}
def show(title, res):
    print(f'  {title:14s} ' + '  '.join(f'{k}={res[k]:.3f}' for k in ['F1', 'FPR', 'AUROC', 'TPR@1%FPR']))
for lang in ['en', 'ru']:
    d = data[lang]
    F = feature_table(d.norm, [lang] * len(d), STATS[lang])
    tr, va, te = split(d, SEED)
    SPLITS[lang], FEATS[lang] = (tr.index, va.index, te.index), F
    res, models = train_eval_all(F.loc[tr.index], tr.label.values, F.loc[te.index], te.label.values)
    print(f'=== {lang}: train={len(tr)} val={len(va)} test={len(te)} ===')
    for name, r in res.items(): show(name, r)
    results['E1'][lang] = res
    HYB[lang] = models['hybrid']
    cols = VARIANTS['hybrid']
    coefs = pd.Series(HYB[lang][-1].coef_[0], index=cols).sort_values(key=abs, ascending=False)
    print('  важнейшие признаки гибрида:', coefs.head(6).round(2).to_dict())
    results['E1'][lang]['hybrid_top_features'] = coefs.head(8).round(3).to_dict()
thr = {l: {'fpr10': threshold_at_fpr(HYB[l], FEATS[l].loc[SPLITS[l][1], VARIANTS['hybrid']].values, data[l].loc[SPLITS[l][1]].label.values, 0.10),
           'fpr02': threshold_at_fpr(HYB[l], FEATS[l].loc[SPLITS[l][1], VARIANTS['hybrid']].values, data[l].loc[SPLITS[l][1]].label.values, 0.02)} for l in HYB}
print('Пороги (валидация): ', thr)
export(HYB, thr, f'{WORK}/weights_hybrid.json', meta={'lm': LM_NAMES, 'max_tokens': 256})

## 6. Э3 — перенос на корпус RuATD (другие генераторы)

In [ ]:
if len(data['coat']):
    Fc = feature_table(data['coat'].norm, ['ru'] * len(data['coat']), STATS['coat'])
    yc = data['coat'].label.values
    tr_idx = SPLITS['ru'][0]; Fr = FEATS['ru']
    res, _ = train_eval_all(Fr.loc[tr_idx], data['ru'].loc[tr_idx].label.values, Fc, yc)
    print(f'=== M4 (ru) -> RuATD: {len(Fc)} текстов ===')
    for name, r in res.items(): show(name, r)
    results['E3']['M4_ru->RuATD'] = res
else:
    print('RuATD недоступен — эксперимент пропущен')

## 7. Э4 — зависимость от длины текста

In [ ]:
import matplotlib.pyplot as plt
LENGTHS = [32, 64, 128, 256]
for lang in ['en', 'ru']:
    d = data[lang]; tr, va, te = SPLITS[lang]; results['E4'][lang] = {}
    for k in LENGTHS:
        F = feature_table(d.norm, [lang] * len(d), STATS[lang], k=k - 1, decode=decode)
        res, _ = train_eval_all(F.loc[tr], d.loc[tr].label.values, F.loc[te], d.loc[te].label.values)
        results['E4'][lang][k] = {v: res[v]['AUROC'] for v in res}
        print(lang, k, {v: round(res[v]['AUROC'], 3) for v in res})
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
names = {'stylometric': 'стилометрический', 'statistical': 'статистический', 'hybrid': 'гибридный'}
for ax, lang in zip(axes, ['en', 'ru']):
    for v in VARIANTS:
        ax.plot(LENGTHS, [results['E4'][lang][k][v] for k in LENGTHS], marker='o', label=names[v])
    ax.set_title('английский' if lang == 'en' else 'русский'); ax.set_xlabel('длина текста, токенов'); ax.set_xscale('log', base=2)
    ax.set_xticks(LENGTHS); ax.set_xticklabels(LENGTHS); ax.grid(alpha=.3)
axes[0].set_ylabel('AUROC'); axes[0].legend()
plt.tight_layout(); plt.savefig(f'{WORK}/length_curve.png', dpi=200); plt.show()

## 8. Э5 — устойчивость к атакам

In [ ]:
from attacks import homoglyph_attack, typo_attack
from stylometry import stylometric_features
from lm_features import aggregate
N_ATTACK = 600
for lang in ['en', 'ru']:
    d = data[lang]; te = SPLITS[lang][2]
    sub = pd.concat([g.sample(min(len(g), N_ATTACK // 2), random_state=SEED) for _, g in d.loc[te].groupby('label')])
    y = sub.label.values; results['E5'][lang] = {}
    cols = VARIANTS['hybrid']
    for attack_name, fn in [('без атаки', None), ('гомоглифы 15%', homoglyph_attack), ('опечатки 10%', typo_attack)]:
        raw = sub.text.astype(str).tolist() if fn is None else [fn(t, seed=i) for i, t in enumerate(sub.text.astype(str))]
        for norm_on in ([True] if fn is None else [False, True]):
            texts = [normalize(t) for t in raw] if norm_on else [t.strip() for t in raw]
            st = get_stats(texts, lang)
            F = pd.DataFrame([{**stylometric_features(t, lang), **aggregate(s)} for t, s in zip(texts, st)])
            p = HYB[lang].predict_proba(F[cols].values)[:, 1]
            from metrics_utils import evaluate
            r = evaluate(y, p, n_boot=0)
            key = f'{attack_name}' + ('' if fn is None else (' + нормализация' if norm_on else ' без нормализации'))
            results['E5'][lang][key] = r
            print(f'{lang} | {key:38s} AUROC={r["AUROC"]:.3f}  F1={r["F1"]:.3f}  Recall={r["Recall"]:.3f}')

## 9. Сохранение результатов

In [ ]:
import json
json.dump(results, open(f'{WORK}/results_step2.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2, default=str)
print('Сохранено в', WORK, ': results_step2.json, weights_hybrid.json, length_curve.png')